# Hinglish Turn Detection — Analysis Walkthrough

This notebook walks through the dataset analysis, training results, and evaluation findings for the Shiprocket voice turn detection challenge.

All numbers come from pre-computed JSON files in `stats/` — you do not need to re-run training to view this notebook.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

BASE = Path('..')
STATS = BASE / 'stats'
PLOTS = STATS / 'plots'

## 1. Dataset Overview

In [ ]:
with open(STATS / 'data_analysis.json') as f:
    da = json.load(f)

bal = da['overall_balance']
print(f"Sample size: {da['sample_size']:,}")
print(f"  Turn-end  (True) : {bal['true']:,}  ({bal['true_pct']:.1f}%)")
print(f"  Mid-turn (False) : {bal['false']:,}  ({bal['false_pct']:.1f}%)")
print(f"  pos_weight       : {bal['pos_weight']:.4f}  (near 1.0 — no reweighting needed)")
print(f"\nHard negatives (midfiller=True AND endpoint=False): {da['hard_negatives']['count']:,}  ({da['hard_negatives']['pct_of_negatives']:.1f}% of negatives)")
print(f"Hindi (hin) rows  : {da['hin_subset']['count']}  (all synthetic TTS — NOT real Hinglish)")

In [ ]:
# Language distribution
lang_df = pd.DataFrame.from_dict(da['language_counts'], orient='index', columns=['count'])
lang_df = lang_df.sort_values('count', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if lang == 'hin' else '#3498db' for lang in lang_df.index]
lang_df['count'].plot(kind='bar', ax=ax, color=colors)
ax.set_title('Top 15 Languages in Training Data (red = hin target)', fontsize=13)
ax.set_ylabel('Row count')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Dataset source breakdown
src_df = pd.DataFrame.from_dict(da['dataset_source_counts'], orient='index', columns=['count'])
src_df = src_df.sort_values('count', ascending=False)

fig, ax = plt.subplots(figsize=(9, 3.5))
src_df['count'].plot(kind='bar', ax=ax, color='#9b59b6')
ax.set_title('Dataset Source Distribution (chirp3 = synthetic TTS)', fontsize=12)
ax.set_ylabel('Row count')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

print(f"\nchirp3_1 + chirp3_2 = {src_df.loc['chirp3_1', 'count'] + src_df.loc['chirp3_2', 'count']:,} rows  ({(src_df.loc['chirp3_1', 'count'] + src_df.loc['chirp3_2', 'count']) / da['sample_size'] * 100:.1f}% of sample)")
print("These are synthetic TTS — model risk: learning voice artifacts, not pure prosody")

## 2. EDA Summary Plot

In [ ]:
img = mpimg.imread(PLOTS / 'eda_summary.png')
fig, ax = plt.subplots(figsize=(16, 9))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Experiment Results Comparison

In [ ]:
results = {}
for exp in [1, 2, 3]:
    with open(STATS / f'exp{exp}_results.json') as f:
        results[exp] = json.load(f)

rows = []
for exp, r in results.items():
    vm = r['best_val_metrics']
    rows.append({
        'Experiment': f'Exp {exp}',
        'Description': ['Baseline (mean pool, frozen)', 'Attention pool + partial unfreeze', 'Hard-neg 3x oversample'][exp-1],
        'Best Epoch': r['best_epoch'],
        'Val AUROC': f"{vm['auroc']:.4f}",
        'Val F1': f"{vm['f1']:.4f}",
        'Val Acc': f"{vm['acc']:.4f}",
        'Trainable %': f"{r['param_info']['trainable_pct']}%"
    })

df = pd.DataFrame(rows)
df.set_index('Experiment', inplace=True)
df

In [ ]:
# Training curves for all experiments
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, exp in enumerate([1, 2, 3]):
    img = mpimg.imread(PLOTS / f'exp{exp}_curves.png')
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f'Experiment {exp}', fontsize=12)
plt.suptitle('Training Curves (loss, val AUROC, val F1)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Full Evaluation — Best Model (Experiment 2)

In [ ]:
with open(STATS / 'exp2_full_eval.json') as f:
    eval2 = json.load(f)

t = eval2['test']
print("Main Test Set (n=500)")
print(f"  Accuracy : {t['accuracy']:.4f}")
print(f"  Precision: {t['precision']:.4f}")
print(f"  Recall   : {t['recall']:.4f}")
print(f"  F1       : {t['f1']:.4f}")
print(f"  AUROC    : {t['auroc']:.4f}")

h = eval2['hinglish']
print("\nHinglish Held-Out Set (n=60, synthetic, out-of-distribution)")
print(f"  Accuracy : {h['hinglish_accuracy']:.4f}")
print(f"  F1       : {h['hinglish_f1']:.4f}")
print(f"  AUROC    : {h['hinglish_auroc']:.4f}")

print(f"\nGap (Hinglish - Main):")
print(f"  AUROC delta : {h['hinglish_auroc'] - t['auroc']:+.4f}")
print(f"  F1 delta    : {h['hinglish_f1'] - t['f1']:+.4f}")

In [ ]:
# Per-language breakdown
lang_metrics = eval2['test']['per_language']
lang_df = pd.DataFrame(lang_metrics).T[['n', 'accuracy', 'f1', 'auroc']]
lang_df.columns = ['n', 'Accuracy', 'F1', 'AUROC']
lang_df = lang_df.sort_values('AUROC', ascending=False)
print("Per-language metrics (Experiment 2, main test set):")
print(lang_df.to_string())
print("\nNote: 'hin' scores high because test clips are same synthetic TTS as training data.")
print("This does NOT reflect real Hinglish performance — see Hinglish held-out set above.")

## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

images = [
    ('exp2_test_cm.png',     'Exp 2 — Main Test Set'),
    ('exp2_hinglish_cm.png', 'Exp 2 — Hinglish Held-Out'),
    ('exp3_test_cm.png',     'Exp 3 — Main Test Set'),
    ('exp3_hinglish_cm.png', 'Exp 3 — Hinglish Held-Out'),
]

for ax, (fname, title) in zip(axes.flat, images):
    img = mpimg.imread(PLOTS / fname)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(title, fontsize=12)

plt.suptitle('Confusion Matrices — Exp 2 vs Exp 3', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Source Leakage Check

In [ ]:
src2 = eval2['test']['per_source']
src_df2 = pd.DataFrame(src2).T[['n', 'accuracy', 'auroc']]
src_df2.columns = ['n', 'Accuracy', 'AUROC']
src_df2 = src_df2.sort_values('AUROC', ascending=False)

print("Per-source breakdown (Experiment 2):")
print(src_df2.to_string())

aurocs = [v['auroc'] for v in src2.values()]
print(f"\nAUROC spread: {max(aurocs) - min(aurocs):.3f}")
print("midcentury_1 AUROC = 0.460 (near-random) — model fails on real human speech")

## 7. Error Analysis

In [ ]:
with open(STATS / 'exp2_errors.json') as f:
    errors2 = json.load(f)

err_df = pd.DataFrame(errors2)
print(f"Total errors analysed: {len(err_df)}")
print(f"  False Positives (predicted turn-end, was mid-turn): {(err_df['error_type']=='FP').sum()}")
print(f"  False Negatives (predicted mid-turn, was turn-end): {(err_df['error_type']=='FN').sum()}")
print()
print("FP breakdown by source:")
fp = err_df[err_df['error_type']=='FP']
print(fp[['language', 'dataset_src', 'midfiller', 'prob']].to_string(index=False))
print()
print("FN breakdown by source:")
fn = err_df[err_df['error_type']=='FN']
print(fn[['language', 'dataset_src', 'midfiller', 'prob']].to_string(index=False))

## 8. ONNX Latency

In [ ]:
lat2 = eval2['latency']
with open(STATS / 'exp3_full_eval.json') as f:
    eval3 = json.load(f)
lat3 = eval3['latency']

lat_df = pd.DataFrame([
    {'Experiment': 'Exp 2', 'Size (MB)': lat2['onnx_size_mb'], 'Mean (ms)': f"{lat2['latency_mean_ms']:.1f}",
     'P95 (ms)': f"{lat2['latency_p95_ms']:.1f}", 'RTF': f"{lat2['rtf']:.4f}"},
    {'Experiment': 'Exp 3', 'Size (MB)': lat3['onnx_size_mb'], 'Mean (ms)': f"{lat3['latency_mean_ms']:.1f}",
     'P95 (ms)': f"{lat3['latency_p95_ms']:.1f}", 'RTF': f"{lat3['rtf']:.4f}"},
]).set_index('Experiment')

print("ONNX CPU Latency (single thread, 100 runs, 2s audio input):")
print(lat_df.to_string())
print("\nBoth models are real-time capable (RTF < 1.0). Exp 2 is ~2.5x faster.")